# 02 — Screening pré-registrado no conjunto já visto de 24/08/2026

Nome deliberado do split: `validation_seen_20260824`. Este dia já orientou a regra H6, portanto serve para depuração e eliminação de candidatas, **não** como prova final de generalização. A matriz é 2×3: verdade humana P/I × saída P/I/ABSTEM.

Critério de segurança proposto: precisão I ≥85% e limite inferior final ≥80%; recall I ≥70%; precisão P sem queda >2 p.p.; coverage ≥80% e sem queda >5 p.p.; falsa acusação ≤5%; número mínimo de alegações I. NaN nunca é convertido em 100%.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

for pasta in (Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks'):
    if (pasta / 'improdutividade_eval.py').is_file():
        sys.path.insert(0, str(pasta))
        break
from improdutividade_eval import (A, I, P, avaliar_guardrails, bootstrap_precision_i_cluster,
    catalogo_candidatas, localizar_dados, matriz_2x3, tabela_metricas)
DATA_ROOT = localizar_dados('unidades_avaliadas.csv', 'predictions_B.csv', 'recontagem_catalogo.csv')
print('Dados carregados: unidades, H6 e recontagem de 24/08')

Dados carregados: unidades, H6 e recontagem de 24/08


In [2]:
unidades = pd.read_csv(DATA_ROOT / 'unidades_avaliadas.csv')
h6 = pd.read_csv(DATA_ROOT / 'predictions_B.csv').rename(columns={'predicao_sistema': 'pred_h6'})
origens = pd.read_csv(DATA_ROOT / 'recontagem_catalogo.csv')[['sample_id', 'nivel_agora', 'motivo_agora']]
df = (unidades.merge(h6[['unit_id', 'pred_h6', 'nivel_h6', 'motivo_h6']], on='unit_id', how='left')
      .merge(origens, left_on='unit_id', right_on='sample_id', how='left'))
df['split'] = 'validation_seen_20260824'
df['pred_legacy'] = df['pred_A']
df['pred_h6'] = df['pred_h6'].fillna(df['pred_B']).fillna(A)

# C1 testa somente a separação de eixos. Não é candidato final: remove toda
# alegação I cuja origem é presença/identidade e não inventa substituta.
df['pred_c1_eixos'] = df['pred_h6']
origem = df['nivel_h6'].fillna(df['nivel_agora']).astype(str).str.lower()
df.loc[(df['pred_c1_eixos'] == I) & origem.isin(['presenca', 'identidade']), 'pred_c1_eixos'] = A

assert set(df['data'].astype(str)) == {'20260824'}, 'Este caderno só pode usar o dia visto de 24/08.'
assert df['y_true'].isin([P, I]).all()
print(f'{len(df)} unidades, {df.video_id.nunique()} vídeos, {df.peso_s.sum()/3600:.2f} h')
print(df['y_true'].value_counts().to_dict())

111 unidades, 43 vídeos, 1.72 h
{'PRODUTIVO': 87, 'IMPRODUTIVO': 24}


In [3]:
politicas = {
    'A_legado': 'pred_legacy',
    'B_H6_atual': 'pred_h6',
    'C1_separar_eixos': 'pred_c1_eixos',
}
metricas = tabela_metricas(df, politicas)
cols_pct = ['precision_I','recall_I','f1_I','precision_P','coverage','abstention',
            'selective_accuracy','operational_accuracy','false_accusation_rate']
exibir = metricas.copy()
for c in cols_pct:
    exibir[c] = exibir[c].map(lambda x: f'{100*x:.2f}%' if pd.notna(x) else 'indefinida')
display(exibir[cols_pct + ['claims_I', 'claims_I_min', 'abstention_min']])

,precision_I,recall_I,f1_I,precision_P,coverage,abstention,selective_accuracy,operational_accuracy,false_accusation_rate,claims_I,claims_I_min,abstention_min
politica,,,,,,,,,,,,
A_legado,27.72%,85.65%,41.88%,92.04%,100.00%,0.00%,51.42%,51.42%,57.37%,70.0,65.305667,0.000000
B_H6_atual,70.65%,56.93%,63.05%,87.85%,65.06%,34.94%,83.50%,54.32%,6.08%,18.0,17.033333,36.133333
C1_separar_eixos,indefinida,0.00%,indefinida,87.85%,48.59%,51.41%,87.85%,42.68%,0.00%,0.0,0.000000,53.166667


In [4]:
print('Matriz H6 — contagem de unidades')
display(matriz_2x3(df, 'pred_h6').astype(int))
print('Matriz H6 — minutos')
display((matriz_2x3(df, 'pred_h6', 'peso_s') / 60).round(3))

lo, hi, n_clusters = bootstrap_precision_i_cluster(df, 'pred_h6', n=4000)
print(f'IC95% exploratório da precisão I, bootstrap por vídeo ({n_clusters} clusters): {lo:.1%} a {hi:.1%}')
print('AVISO: o intervalo continua sendo de um único dia já visto; não é garantia de produção.')

Matriz H6 — contagem de unidades


_pred,PRODUTIVO,IMPRODUTIVO,ABSTEM
_true,,,
PRODUTIVO,47,5,35
IMPRODUTIVO,8,13,3


Matriz H6 — minutos


_pred,PRODUTIVO,IMPRODUTIVO,ABSTEM
_true,,,
PRODUTIVO,44.139,5.000,33.133
IMPRODUTIVO,6.103,12.033,3.000


IC95% exploratório da precisão I, bootstrap por vídeo (43 clusters): 40.0% a 94.7%
AVISO: o intervalo continua sendo de um único dia já visto; não é garantia de produção.


In [5]:
baseline = metricas.loc['B_H6_atual']
gates = pd.DataFrame({nome: avaliar_guardrails(linha, baseline)
                      for nome, linha in metricas.iterrows()}).T
gates['aprovada_sem_IC'] = gates.all(axis=1)
display(gates)
print('Resultado do screening:', 'nenhuma política aprovada' if not gates.aprovada_sem_IC.any() else 'há política candidata')

,precision_I>=85%,recall_I>=70%,precision_P_queda<=2pp,coverage>=80%,coverage_queda<=5pp,false_accusation<=5%,claims_I_suficientes,aprovada_sem_IC
A_legado,False,True,True,True,True,False,True,False
B_H6_atual,False,False,True,False,True,False,False,False
C1_separar_eixos,False,False,True,False,False,True,False,False


Resultado do screening: nenhuma política aprovada


In [6]:
negativas_h6 = df[df['pred_h6'] == I].copy()
display(negativas_h6.groupby(origem.loc[negativas_h6.index])
        .agg(unidades=('unit_id','size'), minutos=('peso_s', lambda s: s.sum()/60)))
print(f"Das {negativas_h6.peso_s.sum()/60:.2f} min alegadas como I pela H6, "
      f"{negativas_h6[origem.loc[negativas_h6.index].isin(['presenca','identidade'])].peso_s.sum()/60:.2f} min vieram de presença/identidade.")
display(catalogo_candidatas())
print('C2–C6 ficam bloqueadas neste export: faltam motivo persistido, quadros/track, interlocutor ou GT da ponte.')

,unidades,minutos
nivel_h6,,
identidade,9,9.000000
presenca,9,8.033333


Das 17.03 min alegadas como I pela H6, 17.03 min vieram de presença/identidade.


,regra,evidencia_necessaria,status_historico
id,,,
C0,baseline vigente,controle,estimável
C1,separar presença/identidade de atividade,nivel/origem,estimável parcialmente
C2,motivo negativo em whitelist,produtividade_motivo,não estimável no histórico
C3,veto por mãos/máquina ativa,mãos+movimento+modo,não estimável end-to-end
C4,persistência 2-de-3 do mesmo motivo,quadros+track+motivo,não estimável no agregado
C5,conversa só com interlocutor confirmado,bbox_stats.interlocutor,não estimável no CSV
C6,resgate de monitoramento/ponte por episódio,movimento+CAM2+janela,não estimável sem GT


C2–C6 ficam bloqueadas neste export: faltam motivo persistido, quadros/track, interlocutor ou GT da ponte.


## Interpretação executiva

- H6 melhora muito o legado, mas sua precisão I (~70,65%) ainda está abaixo do mínimo solicitado e a falsa acusação (~6,08%) ainda excede o gate proposto.
- Bloquear as negativas de presença/identidade elimina todas as alegações I deste dia: precisão fica **indefinida**, recall vai a zero e coverage cai. Isso é segurança, não melhoria comprovada.
- As candidatas mais plausíveis (motivo negativo persistente, veto positivo, conversa confirmada e resgate episódico de ponte/monitoramento) precisam de novo replay com a telemetria preservada.